# CineScope: Oscar Nomination Recognition (Spark MLlib)

**Local Spark only.** Compares weighted Logistic Regression and GBT models for any mapped Oscar nomination.

The workflow requires corrected point-in-time cast/crew features, restricts evaluation to complete historical award cohorts, selects models by validation PR AUC, and locks the decision threshold before final test evaluation.

In [ ]:
from __future__ import annotations

import json
from itertools import product
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

REPO = Path.cwd()
if not (REPO / "src").exists():
    REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))
load_dotenv(REPO / ".env")

from pyspark.ml import Pipeline
from pyspark.ml.classification import GBTClassifier, LogisticRegression
from pyspark.sql import functions as F

from cinescope.ml.evaluation import (
    apply_threshold,
    evaluate_binary_predictions,
    partition_summary,
    select_threshold,
    temporal_split,
    threshold_sweep,
    with_balanced_class_weights,
    with_positive_score,
)
from cinescope.ml.features import (
    available_pre_release_columns,
    build_feature_pipeline,
    leakage_notes,
    prepare_modeling_frame,
    validate_feature_artifact_metadata,
)
from cinescope.paths import get_paths
from cinescope.spark_session import build_spark_session

paths = get_paths(create_dirs=True, validate_mount=True)
spark = build_spark_session(app_name="cinescope-train-awards", paths=paths)
spark.sparkContext.setLogLevel("WARN")

metrics_dir = REPO / "outputs" / "metrics"
charts_dir = REPO / "outputs" / "charts" / "generated"
metrics_dir.mkdir(parents=True, exist_ok=True)
charts_dir.mkdir(parents=True, exist_ok=True)
model_dir = paths.data_root / "models" / "awards_selected_model"


def save_and_show(fig, path: Path):
    fig.savefig(path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("wrote", path)


cast_metrics_path = metrics_dir / "cast_crew_metrics.json"
if not cast_metrics_path.exists():
    raise RuntimeError("Run the corrected cast/crew and Oscar jobs before training")
validate_feature_artifact_metadata(json.loads(cast_metrics_path.read_text()))

raw = spark.read.parquet(str(paths.movies_awards_enriched_dir))
df = prepare_modeling_frame(raw)
feature_cols = available_pre_release_columns(df)
for column in feature_cols:
    df = df.withColumn(column, F.col(column).cast("double"))

train, validation, test = temporal_split(
    df,
    min_year=1960,
    train_end=2007,
    validation_end=2013,
    test_end=2019,
)
train, class_weights = with_balanced_class_weights(
    train,
    label_col="label_awards",
)
train.cache()
validation.cache()
test.cache()

partitions = {
    "train": partition_summary(train, label_col="label_awards"),
    "validation": partition_summary(validation, label_col="label_awards"),
    "test": partition_summary(test, label_col="label_awards"),
}
if any(summary["positives"] == 0 for summary in partitions.values()):
    raise RuntimeError(f"Every temporal partition must contain nominees: {partitions}")

print("feature_cols", feature_cols)
print("class_weights", class_weights)
display(pd.DataFrame.from_dict(partitions, orient="index"))

In [ ]:
oscar_metrics_path = metrics_dir / "oscar_metrics.json"
if not oscar_metrics_path.exists():
    raise RuntimeError("Run the corrected Oscar job before training")
validate_feature_artifact_metadata(json.loads(oscar_metrics_path.read_text()))

In [ ]:
feature_pipeline = build_feature_pipeline(feature_cols, label_col="label_awards")
candidate_specs = []
for reg_param, elastic_net in product([0.01, 0.1], [0.0, 0.5]):
    candidate_specs.append(
        (
            f"lr_reg{reg_param}_enet{elastic_net}",
            "LogisticRegression",
            {"regParam": reg_param, "elasticNetParam": elastic_net, "maxIter": 100},
            LogisticRegression(
                labelCol="label_awards",
                featuresCol="features",
                weightCol="class_weight",
                maxIter=100,
                regParam=reg_param,
                elasticNetParam=elastic_net,
            ),
        )
    )
for max_depth, max_iter in product([3, 5], [20, 40]):
    candidate_specs.append(
        (
            f"gbt_depth{max_depth}_iter{max_iter}",
            "GBTClassifier",
            {"maxDepth": max_depth, "maxIter": max_iter, "stepSize": 0.1},
            GBTClassifier(
                labelCol="label_awards",
                featuresCol="features",
                weightCol="class_weight",
                maxDepth=max_depth,
                maxIter=max_iter,
                stepSize=0.1,
                seed=42,
            ),
        )
    )

candidate_results = []
candidate_models = {}
for candidate_id, algorithm, parameters, estimator in candidate_specs:
    fitted = Pipeline(stages=feature_pipeline.getStages() + [estimator]).fit(train)
    validation_prediction = with_positive_score(fitted.transform(validation)).cache()
    validation_metrics = evaluate_binary_predictions(
        validation_prediction,
        label_col="label_awards",
    )
    candidate_results.append(
        {
            "candidate_id": candidate_id,
            "algorithm": algorithm,
            "parameters": parameters,
            "validation_metrics": validation_metrics,
        }
    )
    candidate_models[candidate_id] = fitted
    validation_prediction.unpersist()
    print(candidate_id, validation_metrics)

selected_candidate = max(
    candidate_results,
    key=lambda result: (
        result["validation_metrics"]["pr_auc"],
        result["validation_metrics"]["roc_auc"],
    ),
)
model = candidate_models[selected_candidate["candidate_id"]]

validation_prediction = with_positive_score(model.transform(validation)).cache()
threshold_results = threshold_sweep(
    validation_prediction,
    label_col="label_awards",
    thresholds=np.linspace(0.01, 0.75, 38).tolist(),
)
selected_threshold = select_threshold(threshold_results)

test_prediction = with_positive_score(model.transform(test)).cache()
test_selected = apply_threshold(
    test_prediction,
    threshold=selected_threshold["threshold"],
)
test_metrics = evaluate_binary_predictions(
    test_selected,
    label_col="label_awards",
    prediction_col="prediction_selected",
)

print("selected_candidate", selected_candidate)
print("selected_threshold", selected_threshold)
print("test_metrics", test_metrics)

In [ ]:
tn, fp = test_metrics["tn"], test_metrics["fp"]
fn, tp = test_metrics["fn"], test_metrics["tp"]
cm = np.array([[tn, fp], [fn, tp]], dtype=float)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

im = axes[0].imshow(cm, cmap="Purples")
axes[0].set_xticks([0, 1], ["pred 0", "pred 1"])
axes[0].set_yticks([0, 1], ["actual 0", "actual 1"])
axes[0].set_title("Oscar model: chronological test")
for (row, column), value in np.ndenumerate(cm):
    axes[0].text(column, row, int(value), ha="center", va="center")
fig.colorbar(im, ax=axes[0], fraction=0.046)

metric_names = ["PR AUC", "ROC AUC", "precision", "recall", "positive F1"]
metric_values = [
    test_metrics["pr_auc"],
    test_metrics["roc_auc"],
    test_metrics["precision"] or 0,
    test_metrics["recall"] or 0,
    test_metrics["positive_f1"] or 0,
]
axes[1].bar(metric_names, metric_values, color=["C4", "C0", "C2", "C3", "C1"])
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=30)
axes[1].set_title(f"Locked threshold = {selected_threshold['threshold']:.3f}")
axes[1].grid(True, axis="y", alpha=0.3)
fig.tight_layout()
save_and_show(fig, charts_dir / "awards_model_confusion_metrics.png")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(
    [row["recall"] for row in threshold_results],
    [row["precision"] for row in threshold_results],
    color="C4",
)
ax.scatter(
    [selected_threshold["recall"]],
    [selected_threshold["precision"]],
    color="C3",
    label=f"selected {selected_threshold['threshold']:.3f}",
)
ax.set(xlabel="Recall", ylabel="Precision", title="Oscar model validation threshold sweep")
ax.grid(True, alpha=0.3)
ax.legend()
save_and_show(fig, charts_dir / "awards_model_pr_curve.png")

comparison = pd.DataFrame(
    [
        {
            "candidate": row["candidate_id"],
            "algorithm": row["algorithm"],
            "PR AUC": row["validation_metrics"]["pr_auc"],
            "ROC AUC": row["validation_metrics"]["roc_auc"],
        }
        for row in candidate_results
    ]
).sort_values("PR AUC", ascending=False)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(comparison["candidate"], comparison["PR AUC"], color="C4")
ax.invert_yaxis()
ax.set(xlabel="Validation PR AUC", title="Oscar model candidate comparison")
ax.grid(True, axis="x", alpha=0.3)
save_and_show(fig, charts_dir / "awards_model_comparison.png")

classifier_model = model.stages[-1]
if hasattr(classifier_model, "featureImportances"):
    importance = np.asarray(classifier_model.featureImportances.toArray())
else:
    importance = np.abs(np.asarray(classifier_model.coefficients.toArray()))
order = np.argsort(importance)[-10:]
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(np.asarray(feature_cols)[order], importance[order], color="C2")
ax.set(xlabel="Model importance", title="Oscar model: top corrected predictors")
ax.grid(True, axis="x", alpha=0.3)
save_and_show(fig, charts_dir / "awards_model_feature_importance.png")

In [ ]:
display(Markdown("### Oscar model: corrected chronological evaluation"))
display(comparison)
display(Markdown("### Complete historical cohorts"))
display(pd.DataFrame.from_dict(partitions, orient="index"))
display(Markdown("### Selected validation operating point"))
display(pd.DataFrame([selected_threshold]))
display(Markdown("### Untouched chronological test"))
display(pd.DataFrame([test_metrics]))

display(Markdown("### Interpretation"))
display(Markdown(
    "- The outcome is any mapped Oscar nomination, not a win or Best Picture nomination.\n"
    "- Candidate and threshold selection use validation years only.\n"
    "- PR AUC and positive-class precision, recall, and F1 are primary because nominations are rare.\n"
    "- Corrected features exclude outcomes, Oscar fields, full-career statistics, and retrospective vote totals."
))
display(pd.DataFrame({"Leakage note": leakage_notes()}))

In [ ]:
model.write().overwrite().save(str(model_dir))
metrics = {
    "schema_version": 2,
    "job": "train_awards_model",
    "model_path": str(model_dir),
    "label": "any mapped Oscar nomination",
    "eligible_release_years": [1960, 2019],
    "feature_cols": feature_cols,
    "leakage": leakage_notes(),
    "selection_rule": "maximum validation PR AUC; threshold maximizes validation positive F1",
    "class_weights": class_weights,
    "partitions": partitions,
    "candidate_results": candidate_results,
    "selected_candidate": selected_candidate,
    "threshold_selection": {
        "source": "validation",
        "selected": selected_threshold,
        "sweep": threshold_results,
    },
    "test_metrics": test_metrics,
    "charts": [
        str(charts_dir / "awards_model_confusion_metrics.png"),
        str(charts_dir / "awards_model_pr_curve.png"),
        str(charts_dir / "awards_model_comparison.png"),
        str(charts_dir / "awards_model_feature_importance.png"),
    ],
    "environment": "local Spark notebook",
    "coverage_note": "The model uses complete historical film cohorts and only Oscar rows mapped to IMDb title IDs.",
}
out = metrics_dir / "awards_model_metrics.json"
out.write_text(json.dumps(metrics, indent=2) + "\n", encoding="utf-8")
print("Saved corrected model + metrics:", out)

validation_prediction.unpersist()
test_prediction.unpersist()
train.unpersist()
validation.unpersist()
test.unpersist()
spark.stop()